# Figure 9

Temperature-scaling fingerprint from the notebook path. This runs the experiment module directly with reduced settings and then plots the scaled collapse.

In [ ]:
from pathlib import Path
import importlib.util
import sys


def find_repo_root():
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "src" / "ghosts").exists() and (base / "experiments").exists():
            return base
    raise RuntimeError("Run this notebook from inside the ghosts-of-softmax repository.")


REPO = find_repo_root()
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


def load_module(name, relative_path):
    path = REPO / relative_path
    module_dir = str(path.parent)
    if module_dir not in sys.path:
        sys.path.insert(0, module_dir)
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


OUTPUT_ROOT = Path('/tmp/ghosts-of-softmax-notebooks')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO}")
print(f"Notebook outputs: {OUTPUT_ROOT}")


In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import torch

temp = load_module('fig9_temp_run', 'experiments/tempfingerprint/run.py')

device = torch.device('cpu')
seeds = [0, 1]
temps = [0.5, 1.0, 2.0, 4.0]
train_steps = 30
train_lr = 0.01
batch_size = 128

data = temp.run_experiment(seeds, temps, train_steps, train_lr, batch_size, device)

out_dir = OUTPUT_ROOT / 'fig9'
out_dir.mkdir(parents=True, exist_ok=True)
out_png = out_dir / 'tempfingerprint.png'
out_pdf = out_dir / 'tempfingerprint.pdf'
out_summary = out_dir / 'summary.json'

temp.make_plot(data, temps, out_png, out_pdf)
summary = temp.build_summary(
    data,
    temps,
    {
        'seeds': seeds,
        'temps': temps,
        'train_steps': train_steps,
        'train_lr': train_lr,
        'batch_size': batch_size,
    },
    out_dir / 'fingerprint_multiseed.pt',
    out_png,
    out_pdf,
    out_summary,
)
out_summary.write_text(json.dumps(summary, indent=2))

display(Image(filename=str(out_png)))
summary['temperatures']
